# Corrigé du TP1 : environnement de travail et première exploration

## Objectifs

- créer un environnement virtuel avec `venv` et y installer les bibliothèques avec `pip`
- lancer JupyterLab depuis cet environnement
- pratiquer les compréhensions, les fonctions `lambda` et les f-strings
- créer des tableaux numpy, calculer et filtrer sans boucle
- créer des `Series` et des `DataFrame` pandas
- lire et écrire des fichiers CSV, Excel et Parquet
- explorer un jeu de données et noter ce qu'il faudra nettoyer

## Livrable

- un environnement virtuel fonctionnel et son fichier `requirements.txt`
- ce notebook complété, avec une exploration commentée du jeu de données : dimensions, types, valeurs aberrantes repérées, premières questions à creuser

## Comment travailler

- Le TP suit le cours : chaque partie se fait juste après la section du cours correspondante.
- Complète les cellules marquées `# TODO`, en utilisant les noms de variables indiqués dans l'énoncé.
- Quand un **résultat attendu** est indiqué, compare-le avec ce que tu obtiens.
- Les cellules qui commencent par `# Vérification` contrôlent ton travail avec `assert` : exécute-les sans les modifier. Si tout est juste, elles affichent `OK` ; sinon, elles s'arrêtent sur une `AssertionError` dont le message indique ce qui ne va pas.
- Les questions **Pour aller plus loin** sont facultatives : fais-les si tu as terminé une partie en avance.

## Partie 0 : mise en place de l'environnement

*À faire après les sections 1 et 2 du cours : environnement de travail et Jupyter.*

Les postes sont sous Ubuntu, avec Python 3.12 déjà installé. Dans un terminal, place-toi dans le dossier de ce TP (`tp01-environnement-pandas`), puis :

1. Crée l'environnement virtuel :
   ```bash
   python3 -m venv .venv
   ```
   Si la commande échoue avec le message `ensurepip is not available`, le paquet Ubuntu `python3.12-venv` manque : il s'installe avec `sudo apt install python3.12-venv` (demande au formateur si tu n'as pas les droits d'administration).
2. Active-le, puis vérifie que `python` désigne bien le Python de l'environnement :
   ```bash
   source .venv/bin/activate
   which python
   ```
   L'invite de commande commence par `(.venv)` et le chemin affiché se termine par `.venv/bin/python`.
3. Installe les bibliothèques :
   ```bash
   python -m pip install --upgrade pip
   python -m pip install jupyterlab pandas numpy openpyxl pyarrow
   ```
4. Fige l'environnement dans un fichier : `python -m pip freeze > requirements.txt`
5. Lance JupyterLab depuis ce même terminal : `jupyter lab`, puis ouvre ce notebook.

Sur un autre système : `python -m venv .venv`, puis `.venv\Scripts\Activate.ps1` sous Windows PowerShell, ou `source .venv/bin/activate` sous macOS.

### Question 0.1

Exécute la cellule suivante, puis la vérification. Le chemin affiché doit se terminer dans le dossier `.venv` de ce TP. Si ce n'est pas le cas, Jupyter n'a pas été lancé depuis l'environnement virtuel : arrête-le avec `Ctrl+C` dans le terminal, active l'environnement, puis relance `jupyter lab`.

In [ ]:
import sys

print(sys.executable)
print(sys.version)


In [2]:
# Vérification
assert ".venv" in sys.executable, "Jupyter n'a pas été lancé depuis l'environnement virtuel du TP"
assert sys.version_info >= (3, 12), "Python 3.12 ou plus récent est attendu"
print("Question 0.1 : OK")


Question 0.1 : OK


### Question 0.2

Lis le contenu du fichier `requirements.txt` dans une variable `contenu`, avec `Path("requirements.txt").read_text()`, et affiche les lignes qui concernent pandas et numpy.

Indice : `contenu.splitlines()` découpe le texte en une liste de lignes, et une compréhension de liste permet de ne garder que celles qui commencent par `pandas` ou `numpy`.

In [ ]:
from pathlib import Path

contenu = Path("requirements.txt").read_text()
[ligne for ligne in contenu.splitlines() if ligne.startswith(("pandas", "numpy"))]


In [4]:
# Vérification
assert "pandas==" in contenu, "pandas n'apparaît pas dans requirements.txt : as-tu lancé pip freeze après l'installation ?"
assert "numpy==" in contenu, "numpy n'apparaît pas dans requirements.txt"
print("Question 0.2 : OK")


Question 0.2 : OK


## Partie 1 : Python intermédiaire

*À faire après la section 3 du cours : compréhensions, lambda, f-strings et assert.*

### Question 1.1

Écris une compréhension de liste qui, à partir de `range(1, 21)`, ne garde que les nombres pairs et multiplie-les par 2. Range le résultat dans une variable `pairs_doubles` et affiche-la.

**Résultat attendu** :

```text
[4, 8, 12, 16, 20, 24, 28, 32, 36, 40]
```

In [5]:
pairs_doubles = [n * 2 for n in range(1, 21) if n % 2 == 0]
pairs_doubles


[4, 8, 12, 16, 20, 24, 28, 32, 36, 40]

In [6]:
# Vérification
assert pairs_doubles == [4, 8, 12, 16, 20, 24, 28, 32, 36, 40], "la liste doit contenir les nombres pairs de 1 à 20, multipliés par 2"
print("Question 1.1 : OK")


Question 1.1 : OK


### Question 1.2

À partir de la liste `notes`, construis avec une compréhension la liste `resultats`, qui contient `admis` pour une note supérieure ou égale à 10 et `ajourné` sinon.

Attention : ici, on transforme chaque élément sans en retirer aucun. Où se place le `if` ?

**Résultat attendu** :

```text
['admis', 'ajourné', 'admis', 'ajourné', 'admis', 'admis']
```

In [7]:
notes = [12, 8, 15, 9.5, 10, 18]

resultats = ["admis" if note >= 10 else "ajourné" for note in notes]
resultats


['admis', 'ajourné', 'admis', 'ajourné', 'admis', 'admis']

In [8]:
# Vérification
assert len(resultats) == len(notes), "la liste doit avoir un élément par note"
assert resultats == ["admis", "ajourné", "admis", "ajourné", "admis", "admis"], "vérifie la condition : 10 est une note suffisante"
print("Question 1.2 : OK")


Question 1.2 : OK


### Question 1.3

Écris une compréhension de dictionnaire `longueurs` qui associe chaque mot à sa longueur, uniquement pour les mots de plus de 3 lettres.

**Résultat attendu** :

```text
{'chat': 4, 'table': 5, 'python': 6, 'data': 4}
```

In [9]:
mots = ["chat", "vin", "table", "un", "python", "data"]

longueurs = {mot: len(mot) for mot in mots if len(mot) > 3}
longueurs


{'chat': 4, 'table': 5, 'python': 6, 'data': 4}

In [10]:
# Vérification
assert longueurs == {"chat": 4, "table": 5, "python": 6, "data": 4}, "seuls les mots de plus de 3 lettres doivent être gardés"
print("Question 1.3 : OK")


Question 1.3 : OK


### Question 1.4

À partir du dictionnaire `prix_ht`, construis avec une compréhension le dictionnaire `prix_ttc` : mêmes clés, prix multipliés par 1,2 et arrondis à 2 décimales.

**Résultat attendu** :

```text
{'clavier': 54.0, 'souris': 23.88, 'écran': 226.8, 'câble': 8.99}
```

In [11]:
prix_ht = {"clavier": 45.0, "souris": 19.9, "écran": 189.0, "câble": 7.49}

prix_ttc = {produit: round(prix * 1.2, 2) for produit, prix in prix_ht.items()}
prix_ttc


{'clavier': 54.0, 'souris': 23.88, 'écran': 226.8, 'câble': 8.99}

In [12]:
# Vérification
assert prix_ttc == {"clavier": 54.0, "souris": 23.88, "écran": 226.8, "câble": 8.99}, "vérifie le calcul et l'arrondi à 2 décimales"
print("Question 1.4 : OK")


Question 1.4 : OK


### Question 1.5

Les régions ci-dessous ont été saisies à la main. Construis avec une compréhension d'ensemble l'ensemble `regions_uniques` des régions sans doublon, après avoir retiré les espaces superflus (`strip`) et tout mis en minuscules (`lower`). Affiche-le trié avec `sorted`, car l'ordre d'affichage d'un ensemble n'est pas garanti.

**Résultat attendu** :

```text
['bretagne', 'occitanie', 'île-de-france']
```

In [13]:
saisies = ["Bretagne", "bretagne ", "Occitanie", " occitanie", "Île-de-France", "BRETAGNE"]

regions_uniques = {r.strip().lower() for r in saisies}
sorted(regions_uniques)


['bretagne', 'occitanie', 'île-de-france']

In [14]:
# Vérification
assert regions_uniques == {"bretagne", "occitanie", "île-de-france"}, "il doit rester trois régions, en minuscules et sans espace"
print("Question 1.5 : OK")


Question 1.5 : OK


### Question 1.6

Trie la liste de tuples `(nom, age)` par âge croissant, avec `sorted` et une fonction `lambda`, dans une variable `par_age`. Trouve ensuite la personne la plus âgée avec `max`, dans une variable `doyen`.

**Résultat attendu** :

```text
[('Bob', 22), ('Chloé', 29), ('Alice', 34), ('David', 41)]
('David', 41)
```

In [15]:
personnes = [("Alice", 34), ("Bob", 22), ("Chloé", 29), ("David", 41)]

par_age = sorted(personnes, key=lambda p: p[1])
doyen = max(personnes, key=lambda p: p[1])
print(par_age)
print(doyen)


[('Bob', 22), ('Chloé', 29), ('Alice', 34), ('David', 41)]
('David', 41)


In [16]:
# Vérification
assert par_age == [("Bob", 22), ("Chloé", 29), ("Alice", 34), ("David", 41)], "le tri doit se faire sur l'âge"
assert doyen == ("David", 41), "max doit comparer les âges"
print("Question 1.6 : OK")


Question 1.6 : OK


### Question 1.7

Le catalogue ci-dessous est une liste de dictionnaires, le format habituel des données renvoyées par une API.

1. Trie le catalogue du plus cher au moins cher (`sorted`, `key` et `reverse=True`), dans une variable `catalogue_trie`.
2. Construis la liste `noms_en_stock` des noms des produits en stock (stock supérieur à 0), du plus cher au moins cher.

**Résultat attendu** :

```text
['écran', 'casque', 'clavier']
```

In [17]:
catalogue = [
    {"nom": "clavier", "prix": 45.0, "stock": 12},
    {"nom": "souris", "prix": 19.9, "stock": 0},
    {"nom": "écran", "prix": 189.0, "stock": 3},
    {"nom": "casque", "prix": 79.0, "stock": 5},
    {"nom": "webcam", "prix": 59.9, "stock": 0},
]

catalogue_trie = sorted(catalogue, key=lambda p: p["prix"], reverse=True)
noms_en_stock = [p["nom"] for p in catalogue_trie if p["stock"] > 0]
noms_en_stock


['écran', 'casque', 'clavier']

In [18]:
# Vérification
assert [p["nom"] for p in catalogue_trie] == ["écran", "casque", "webcam", "clavier", "souris"], "le tri doit se faire sur le prix, du plus cher au moins cher"
assert noms_en_stock == ["écran", "casque", "clavier"], "seuls les produits en stock doivent être gardés"
print("Question 1.7 : OK")


Question 1.7 : OK


### Question 1.8

Les prix ci-dessous ont été saisis à la main, certains avec une virgule comme séparateur décimal. Construis avec une compréhension la liste `prix` des prix convertis en nombres décimaux (`float`), puis calcule leur total dans une variable `total`.

**Résultat attendu** :

```text
[12.5, 8.0, 12.5, 199.99, 45.0]
277.99
```

In [19]:
prix_saisis = ["12.5", "8", "12,5", "199,99", "45.00"]

prix = [float(p.replace(",", ".")) for p in prix_saisis]
total = sum(prix)
print(prix)
print(total)


[12.5, 8.0, 12.5, 199.99, 45.0]
277.99


In [20]:
# Vérification
assert prix == [12.5, 8.0, 12.5, 199.99, 45.0], "chaque prix doit être converti en float, virgule remplacée par un point"
assert round(total, 2) == 277.99, "le total doit valoir 277.99"
print("Question 1.8 : OK")


Question 1.8 : OK


### Question 1.9

Avec les spécifications de format des f-strings, construis la variable `phrase` à partir de `ca` et `taux`, pour obtenir exactement le texte attendu : le chiffre d'affaires avec un séparateur de milliers et 2 décimales, le taux en pourcentage avec une décimale.

**Résultat attendu** :

```text
Chiffre d'affaires : 1,234,567.89 euros, taux de réalisation : 45.7%
```

In [21]:
ca = 1234567.891
taux = 0.4567

phrase = f"Chiffre d'affaires : {ca:,.2f} euros, taux de réalisation : {taux:.1%}"
print(phrase)


Chiffre d'affaires : 1,234,567.89 euros, taux de réalisation : 45.7%


In [22]:
# Vérification
assert phrase == "Chiffre d'affaires : 1,234,567.89 euros, taux de réalisation : 45.7%", "compare caractère par caractère avec le résultat attendu"
print("Question 1.9 : OK")


Question 1.9 : OK


### Question 1.10

Écris une fonction `montant_ttc(montant_ht, taux_tva=0.2)` qui renvoie le montant toutes taxes comprises, arrondi à 2 décimales. Écris ensuite, dans la même cellule, trois `assert` qui la vérifient : le cas par défaut, un taux réduit de 5,5 %, et un montant nul.

La cellule de vérification contrôle ta fonction avec ses propres exemples.

In [23]:
def montant_ttc(montant_ht, taux_tva=0.2):
    return round(montant_ht * (1 + taux_tva), 2)


assert montant_ttc(100) == 120.0
assert montant_ttc(100, 0.055) == 105.5
assert montant_ttc(0) == 0


In [24]:
# Vérification
assert montant_ttc(45) == 54.0, "montant_ttc(45) doit valoir 54.0"
assert montant_ttc(19.9) == 23.88, "le résultat doit être arrondi à 2 décimales"
assert montant_ttc(200, taux_tva=0.1) == 220.0, "le taux doit pouvoir être passé en paramètre"
print("Question 1.10 : OK")


Question 1.10 : OK


### Pour aller plus loin 1.11

La fonction `zip` parcourt deux listes en parallèle. Construis le dictionnaire `ventes` qui associe chaque produit à sa quantité, puis, avec une compréhension, le dictionnaire `grosses_ventes` des produits vendus à au moins 5 exemplaires.

**Résultat attendu** :

```text
{'clavier': 3, 'souris': 12, 'écran': 1, 'casque': 7}
{'souris': 12, 'casque': 7}
```

In [25]:
produits = ["clavier", "souris", "écran", "casque"]
quantites = [3, 12, 1, 7]

ventes = dict(zip(produits, quantites))
grosses_ventes = {produit: qte for produit, qte in zip(produits, quantites) if qte >= 5}
print(ventes)
print(grosses_ventes)


{'clavier': 3, 'souris': 12, 'écran': 1, 'casque': 7}
{'souris': 12, 'casque': 7}


In [26]:
# Vérification
assert ventes == {"clavier": 3, "souris": 12, "écran": 1, "casque": 7}
assert grosses_ventes == {"souris": 12, "casque": 7}
print("Question 1.11 : OK")


Question 1.11 : OK


## Partie 2 : numpy

*À faire après la section 4 du cours : les bases de numpy.*

### Question 2.1

Compare le résultat de `[1, 2, 3] * 2` et de `np.array([1, 2, 3]) * 2`, puis explique la différence dans la cellule Markdown suivante.

**Résultat attendu** :

```text
[1, 2, 3, 1, 2, 3]
[2 4 6]
```

In [27]:
import numpy as np

print([1, 2, 3] * 2)
print(np.array([1, 2, 3]) * 2)


[1, 2, 3, 1, 2, 3]
[2 4 6]


**Réponse** : Avec une liste, `* 2` répète la liste : on obtient `[1, 2, 3, 1, 2, 3]`. Avec un tableau numpy, le calcul s'applique à chaque élément : on obtient `[2 4 6]`.

### Question 2.2

Crée les tableaux suivants avec les fonctions de numpy, sans écrire les valeurs à la main, et affiche-les :

- `zeros` : 5 zéros (`np.zeros`) ;
- `uns` : 3 uns (`np.ones`) ;
- `chiffres` : les entiers de 0 à 9 (`np.arange`) ;
- `graduations` : 5 valeurs régulièrement espacées de 0 à 1 inclus (`np.linspace`).

**Résultat attendu** :

```text
[0. 0. 0. 0. 0.]
[1. 1. 1.]
[0 1 2 3 4 5 6 7 8 9]
[0.   0.25 0.5  0.75 1.  ]
```

In [28]:
zeros = np.zeros(5)
uns = np.ones(3)
chiffres = np.arange(10)
graduations = np.linspace(0, 1, 5)

print(zeros)
print(uns)
print(chiffres)
print(graduations)


[0. 0. 0. 0. 0.]
[1. 1. 1.]
[0 1 2 3 4 5 6 7 8 9]
[0.   0.25 0.5  0.75 1.  ]


In [29]:
# Vérification
assert np.array_equal(zeros, [0, 0, 0, 0, 0]), "zeros : 5 zéros"
assert np.array_equal(uns, [1, 1, 1]), "uns : 3 uns"
assert np.array_equal(chiffres, range(10)), "chiffres : de 0 à 9, la fin de arange est exclue"
assert np.allclose(graduations, [0, 0.25, 0.5, 0.75, 1]), "graduations : linspace inclut les deux bornes"
print("Question 2.2 : OK")


Question 2.2 : OK


### Question 2.3

Crée le tableau `notes` à partir de la liste `[12, 8, 15, 9, 18, 14]`. Affiche ensuite :

1. sa forme (`shape`) et le type de ses éléments (`dtype`) ;
2. la première note, puis la dernière ;
3. les trois premières notes.

**Résultat attendu** :

```text
(6,) int64
12 14
[12  8 15]
```

In [30]:
notes = np.array([12, 8, 15, 9, 18, 14])

print(notes.shape, notes.dtype)
print(notes[0], notes[-1])
print(notes[:3])


(6,) int64
12 14
[12  8 15]


In [31]:
# Vérification
assert notes.shape == (6,), "le tableau notes doit contenir 6 valeurs"
print("Question 2.3 : OK")


Question 2.3 : OK


### Question 2.4

Sans boucle :

1. crée `notes_bonus`, les notes augmentées d'un point ;
2. calcule `moyenne`, la moyenne des notes, ainsi que la plus petite et la plus grande note.

**Résultat attendu** :

```text
[13  9 16 10 19 15]
12.666666666666666 8 18
```

In [32]:
notes_bonus = notes + 1
moyenne = notes.mean()

print(notes_bonus)
print(moyenne, notes.min(), notes.max())


[13  9 16 10 19 15]
12.666666666666666 8 18


In [33]:
# Vérification
assert np.array_equal(notes_bonus, [13, 9, 16, 10, 19, 15]), "chaque note doit gagner un point"
assert round(moyenne, 2) == 12.67, "la moyenne des notes vaut 12,67"
print("Question 2.4 : OK")


Question 2.4 : OK


### Question 2.5

Un panier contient trois articles. Calcule, sans boucle :

1. `montants` : le montant de chaque article, prix multiplié par quantité ;
2. `total` : le total du panier.

**Résultat attendu** :

```text
[30. 25. 80.]
135.0
```

In [34]:
prix = np.array([10.0, 25.0, 8.0])
quantites = np.array([3, 1, 10])

montants = prix * quantites
total = montants.sum()
print(montants)
print(total)


[30. 25. 80.]
135.0


In [35]:
# Vérification
assert np.array_equal(montants, [30, 25, 80]), "les deux tableaux se multiplient élément par élément"
assert total == 135, "le total du panier vaut 135"
print("Question 2.5 : OK")


Question 2.5 : OK


### Question 2.6

Avec le tableau `notes` :

1. affiche le résultat de `notes >= 10` ;
2. crée `admises`, le tableau des notes supérieures ou égales à 10 ;
3. calcule `nb_admis`, le nombre de notes supérieures ou égales à 10.

**Résultat attendu** :

```text
[ True False  True False  True  True]
[12 15 18 14]
4
```

In [36]:
print(notes >= 10)
admises = notes[notes >= 10]
nb_admis = (notes >= 10).sum()
print(admises)
print(nb_admis)


[ True False  True False  True  True]
[12 15 18 14]
4


In [37]:
# Vérification
assert np.array_equal(admises, [12, 15, 18, 14]), "admises : les notes supérieures ou égales à 10"
assert nb_admis == 4, "4 notes sont supérieures ou égales à 10"
print("Question 2.6 : OK")


Question 2.6 : OK


### Question 2.7

Le tableau `ventes` contient les ventes de 2 magasins (une ligne par magasin) sur 3 jours (une colonne par jour). Affiche :

1. sa forme ;
2. les ventes du premier magasin ;
3. les ventes du deuxième magasin le troisième jour ;
4. le total par jour (`axis=0`), puis le total par magasin (`axis=1`).

**Résultat attendu** :

```text
(2, 3)
[3 5 2]
6
[7 6 8]
[10 11]
```

In [38]:
ventes = np.array([[3, 5, 2], [4, 1, 6]])

print(ventes.shape)
print(ventes[0])
print(ventes[1, 2])
print(ventes.sum(axis=0))
print(ventes.sum(axis=1))


(2, 3)
[3 5 2]
6
[7 6 8]
[10 11]


### Question 2.8

Crée un générateur de nombres aléatoires avec la seed 0, puis simule 10 lancers de dé dans un tableau `des` :

```python
rng = np.random.default_rng(0)
des = rng.integers(1, 7, size=10)
```

Affiche `des` et sa moyenne. Exécute ensuite la cellule une deuxième fois : obtiens-tu les mêmes valeurs ? Pourquoi ?

**Résultat attendu** :

```text
[6 4 4 2 2 1 1 1 2 5]
2.8
```

In [39]:
rng = np.random.default_rng(0)
des = rng.integers(1, 7, size=10)
print(des)
print(des.mean())


[6 4 4 2 2 1 1 1 2 5]
2.8


In [40]:
# Vérification
assert len(des) == 10 and des.min() >= 1 and des.max() <= 6, "10 entiers entre 1 et 6"
print("Question 2.8 : OK")


Question 2.8 : OK


**Réponse** : Oui : le générateur est recréé avec la même seed au début de la cellule, il produit donc les mêmes nombres. C'est ce qui rend un tirage reproductible. Sans seed, les valeurs changeraient à chaque exécution.

### Pour aller plus loin 2.9

Simule 6 000 lancers de dé avec `rng = np.random.default_rng(42)`, dans un tableau `lancers`. Calcule la proportion de 6 obtenus, `frequence_six`, avec `(lancers == 6).mean()`. Est-elle proche de 1/6, soit environ 0,167 ?

**Résultat attendu** :

```text
0.167
```

In [41]:
rng = np.random.default_rng(42)
lancers = rng.integers(1, 7, size=6000)
frequence_six = (lancers == 6).mean()
print(round(frequence_six, 3))


0.167


In [42]:
# Vérification
assert len(lancers) == 6000, "6 000 lancers sont attendus"
assert abs(frequence_six - 1 / 6) < 0.02, "la fréquence de 6 doit être proche de 1/6"
print("Question 2.9 : OK")


Question 2.9 : OK


## Partie 3 : Series et DataFrame

*À faire après la section 5 du cours : introduction à pandas.*

### Question 3.1

Crée la `Series` `temperatures` des températures 12.5, 14.0, 9.8, 16.2 et 11.1, avec pour index les jours de `lundi` à `vendredi`, et pour nom `temperature`.

**Résultat attendu** :

```text
lundi       12.5
mardi       14.0
mercredi     9.8
jeudi       16.2
vendredi    11.1
Name: temperature, dtype: float64
```

In [43]:
import pandas as pd

temperatures = pd.Series(
    [12.5, 14.0, 9.8, 16.2, 11.1],
    index=["lundi", "mardi", "mercredi", "jeudi", "vendredi"],
    name="temperature",
)
temperatures


lundi       12.5
mardi       14.0
mercredi     9.8
jeudi       16.2
vendredi    11.1
Name: temperature, dtype: float64

In [44]:
# Vérification
assert isinstance(temperatures, pd.Series), "temperatures doit être une Series"
assert list(temperatures.index) == ["lundi", "mardi", "mercredi", "jeudi", "vendredi"], "vérifie l'index"
assert temperatures.name == "temperature"
print("Question 3.1 : OK")


Question 3.1 : OK


### Question 3.2

Accède à la température du mercredi par son label (`.loc`), puis par sa position (`.iloc`).

**Résultat attendu** :

```text
9.8
9.8
```

In [45]:
print(temperatures.loc["mercredi"])
print(temperatures.iloc[2])


9.8
9.8


### Question 3.3

Calcule la température moyenne de la semaine, arrondie à 2 décimales, et trouve le jour le plus chaud avec la méthode `idxmax`. Affiche ensuite les températures supérieures à 12 degrés, avec un filtre booléen comme sur un tableau numpy.

**Résultat attendu** :

```text
12.72
jeudi
lundi    12.5
mardi    14.0
jeudi    16.2
Name: temperature, dtype: float64
```

In [46]:
print(round(temperatures.mean(), 2))
print(temperatures.idxmax())
temperatures[temperatures > 12]


12.72
jeudi


lundi    12.5
mardi    14.0
jeudi    16.2
Name: temperature, dtype: float64

### Question 3.4

Additionne les deux Series ci-dessous. Pourquoi obtient-on des `NaN` ? Utilise ensuite la méthode `add` avec son paramètre `fill_value` pour obtenir un total sans `NaN`, dans une variable `total_ventes`.

**Résultat attendu** :

```text
clavier    NaN
souris     7.0
écran      NaN
dtype: float64
clavier    3.0
souris     7.0
écran      1.0
dtype: float64
```

In [47]:
ventes_lundi = pd.Series({"clavier": 3, "souris": 5})
ventes_mardi = pd.Series({"souris": 2, "écran": 1})

print(ventes_lundi + ventes_mardi)
total_ventes = ventes_lundi.add(ventes_mardi, fill_value=0)
total_ventes


clavier    NaN
souris     7.0
écran      NaN
dtype: float64


clavier    3.0
souris     7.0
écran      1.0
dtype: float64

In [48]:
# Vérification
assert total_ventes.to_dict() == {"clavier": 3, "souris": 7, "écran": 1}, "aucune valeur ne doit manquer"
print("Question 3.4 : OK")


Question 3.4 : OK


**Réponse** : L'addition de deux Series s'aligne sur l'index, pas sur la position. Le clavier n'existe pas le mardi et l'écran pas le lundi : pandas ne trouve rien à additionner et renvoie `NaN`. Avec `fill_value=0`, un label absent d'un côté est traité comme 0.

### Question 3.5

Crée le DataFrame `produits` à partir d'un dictionnaire de listes, avec les colonnes et les valeurs suivantes :

| produit | prix | quantite |
|---|---|---|
| clavier | 45.0 | 3 |
| souris | 19.9 | 10 |
| écran | 189.0 | 1 |
| casque | 79.0 | 2 |
| webcam | 59.9 | 4 |

Ajoute une colonne `montant` égale au prix multiplié par la quantité, puis affiche le tableau.

**Résultat attendu** :

```text
   produit   prix  quantite  montant
0  clavier   45.0         3    135.0
1   souris   19.9        10    199.0
2    écran  189.0         1    189.0
3   casque   79.0         2    158.0
4   webcam   59.9         4    239.6
```

In [49]:
produits = pd.DataFrame({
    "produit": ["clavier", "souris", "écran", "casque", "webcam"],
    "prix": [45.0, 19.9, 189.0, 79.0, 59.9],
    "quantite": [3, 10, 1, 2, 4],
})
produits["montant"] = produits["prix"] * produits["quantite"]
produits


,produit,prix,quantite,montant
0,clavier,45.0,3,135.0
1,souris,19.9,10,199.0
2,écran,189.0,1,189.0
3,casque,79.0,2,158.0
4,webcam,59.9,4,239.6


In [50]:
# Vérification
assert produits.shape == (5, 4), "le DataFrame doit avoir 5 lignes et 4 colonnes"
assert list(produits.columns) == ["produit", "prix", "quantite", "montant"]
assert round(produits["montant"].sum(), 2) == 920.6, "vérifie le calcul de montant"
print("Question 3.5 : OK")


Question 3.5 : OK


### Question 3.6

Sur `produits`, affiche le montant total, le prix moyen arrondi à 2 décimales, et la part de chaque produit dans le montant total, en pourcentage arrondi à 1 décimale, dans une nouvelle colonne `part_pct`.

**Résultat attendu** :

```text
920.6
78.56
   produit  montant  part_pct
0  clavier    135.0      14.7
1   souris    199.0      21.6
2    écran    189.0      20.5
3   casque    158.0      17.2
4   webcam    239.6      26.0
```

In [51]:
print(round(produits["montant"].sum(), 2))
print(round(produits["prix"].mean(), 2))
produits["part_pct"] = (produits["montant"] / produits["montant"].sum() * 100).round(1)
produits[["produit", "montant", "part_pct"]]


920.6
78.56


,produit,montant,part_pct
0,clavier,135.0,14.7
1,souris,199.0,21.6
2,écran,189.0,20.5
3,casque,158.0,17.2
4,webcam,239.6,26.0


In [52]:
# Vérification
assert abs(produits["part_pct"].sum() - 100) < 0.5, "les parts doivent totaliser environ 100 %"
print("Question 3.6 : OK")


Question 3.6 : OK


### Question 3.7

Quelle est la différence entre `produits["prix"]` et `produits[["prix"]]` ? Vérifie avec `type()`.

**Résultat attendu** :

```text
<class 'pandas.Series'>
<class 'pandas.DataFrame'>
```

In [53]:
print(type(produits["prix"]))
print(type(produits[["prix"]]))


<class 'pandas.Series'>
<class 'pandas.DataFrame'>


**Réponse** : Avec un nom de colonne, on obtient une `Series`. Avec une liste de noms, même d'un seul élément, on obtient un `DataFrame`.

### Question 3.8

Les données arrivent souvent sous forme d'une liste de dictionnaires, un par ligne. Crée le DataFrame `commandes` à partir de la liste ci-dessous, puis affiche sa forme et ses colonnes. Que vaut la remise de la deuxième commande, absente de son dictionnaire ?

**Résultat attendu** :

```text
(3, 4)
['id', 'client', 'montant', 'remise']
   id  client  montant  remise
0   1  Durand    120.0    0.10
1   2  Martin     75.5     NaN
2   3   Petit    210.0    0.05
```

In [54]:
lignes = [
    {"id": 1, "client": "Durand", "montant": 120.0, "remise": 0.1},
    {"id": 2, "client": "Martin", "montant": 75.5},
    {"id": 3, "client": "Petit", "montant": 210.0, "remise": 0.05},
]

commandes = pd.DataFrame(lignes)
print(commandes.shape)
print(list(commandes.columns))
commandes


(3, 4)
['id', 'client', 'montant', 'remise']


,id,client,montant,remise
0,1,Durand,120.0,0.10
1,2,Martin,75.5,NaN
2,3,Petit,210.0,0.05


In [55]:
# Vérification
assert commandes.shape == (3, 4)
assert commandes["remise"].isna().sum() == 1, "la remise absente doit être une valeur manquante"
print("Question 3.8 : OK")


Question 3.8 : OK


**Réponse** : pandas crée la colonne `remise` pour toutes les lignes et met `NaN` là où la clé est absente : la valeur est manquante.

### Question 3.9

Un DataFrame peut aussi se construire à partir d'un tableau numpy à deux dimensions. Avec `rng = np.random.default_rng(1)`, tire une grille d'entiers entre 0 et 99 de 4 lignes et 3 colonnes (`rng.integers(0, 100, size=(4, 3))`), et construis le DataFrame `visites` avec pour colonnes `lundi`, `mardi`, `mercredi` et pour index `A`, `B`, `C`, `D` (quatre magasins).

Affiche ensuite le total par jour (`sum()`), puis le total par magasin (`sum(axis=1)`).

**Résultat attendu** :

```text
   lundi  mardi  mercredi
A     47     51        75
B     95      3        14
C     82     94        24
D     31     86        42
lundi       255
mardi       234
mercredi    155
dtype: int64
A    173
B    112
C    200
D    159
dtype: int64
```

In [56]:
rng = np.random.default_rng(1)
visites = pd.DataFrame(
    rng.integers(0, 100, size=(4, 3)),
    columns=["lundi", "mardi", "mercredi"],
    index=["A", "B", "C", "D"],
)
print(visites)
print(visites.sum())
print(visites.sum(axis=1))


   lundi  mardi  mercredi
A     47     51        75
B     95      3        14
C     82     94        24
D     31     86        42
lundi       255
mardi       234
mercredi    155
dtype: int64
A    173
B    112
C    200
D    159
dtype: int64


In [57]:
# Vérification
assert visites.shape == (4, 3)
assert list(visites.columns) == ["lundi", "mardi", "mercredi"] and list(visites.index) == ["A", "B", "C", "D"]
print("Question 3.9 : OK")


Question 3.9 : OK


## Partie 4 : lecture et écriture de fichiers

*À faire après la section 6 du cours : lecture et écriture de fichiers.*

### Question 4.1

Charge le fichier `ressources/ventes.csv` dans une variable `df` avec `pd.read_csv`, et affiche les 5 premières lignes.

In [58]:
df = pd.read_csv("ressources/ventes.csv")
df.head()


,id_commande,date,region,categorie,produit,Prix unitaire,quantite,Client fidèle
0,1021,2025-01-01,Nouvelle-Aquitaine,Bureau,Lampe LED,268.78,9,False
1,1035,2025-01-02,Bretagne,Bureau,Rangement papier,316.64,3,False
2,1044,2025-01-05,Hauts-de-France,Électroménager,Grille-pain,248.61,10,False
3,1026,2025-01-11,Occitanie,Électroménager,Cafetière,92.78,2,False
4,1053,2025-01-13,Bretagne,Bureau,Bureau assis-debout,341.86,3,True


In [59]:
# Vérification
assert df.shape == (62, 8), "le fichier contient 62 lignes et 8 colonnes"
print("Question 4.1 : OK")


Question 4.1 : OK


### Question 4.2

Affiche le type de chaque colonne. La colonne `date` est-elle reconnue comme une date ? Recharge le fichier dans `df` avec l'option qui convient pour qu'elle le soit, et vérifie.

In [60]:
print(df.dtypes)

df = pd.read_csv("ressources/ventes.csv", parse_dates=["date"])
df.dtypes


id_commande        int64
date                 str
region               str
categorie            str
produit              str
Prix unitaire    float64
quantite           int64
Client fidèle       bool
dtype: object


id_commande               int64
date             datetime64[us]
region                      str
categorie                   str
produit                     str
Prix unitaire           float64
quantite                  int64
Client fidèle              bool
dtype: object

In [61]:
# Vérification
assert pd.api.types.is_datetime64_any_dtype(df["date"]), "la colonne date doit être de type datetime64"
print("Question 4.2 : OK")


Question 4.2 : OK


### Question 4.3

Sans charger tout le fichier, lis dans `extrait` seulement les colonnes `date`, `produit` et `quantite` des 10 premières lignes (options `usecols` et `nrows`). Affiche sa forme et ses colonnes.

**Résultat attendu** :

```text
(10, 3)
['date', 'produit', 'quantite']
```

In [62]:
extrait = pd.read_csv("ressources/ventes.csv", usecols=["date", "produit", "quantite"], nrows=10)
print(extrait.shape)
print(list(extrait.columns))


(10, 3)
['date', 'produit', 'quantite']


In [63]:
# Vérification
assert extrait.shape == (10, 3), "10 lignes et 3 colonnes"
print("Question 4.3 : OK")


Question 4.3 : OK


### Question 4.4

Le fichier `ressources/export_magasins.csv` a été exporté depuis un tableur français. Ouvre-le d'abord dans JupyterLab (double-clic dans l'explorateur de fichiers) pour regarder son format.

1. Lis-le sans aucune option dans `brut_magasins` et affiche sa forme et ses colonnes. Que se passe-t-il ?
2. Relis-le en précisant seulement le séparateur de colonnes, et affiche les types. Quelles colonnes ont un type incorrect ?
3. Relis-le dans `magasins` avec toutes les options qui conviennent : séparateur, séparateur décimal, et code postal lu comme du texte pour garder le zéro initial. Affiche `magasins` et ses types.

Ce fichier est encodé en UTF-8. Les exports de tableurs sont souvent encodés en `cp1252` : si une lecture échoue avec une `UnicodeDecodeError`, ajoute `encoding="cp1252"`.

**Résultat attendu** :

```text
(8, 1)
['code_magasin;ville;code_postal;surface_m2;ca_2024']
code_magasin      str
ville             str
code_postal     int64
surface_m2        str
ca_2024           str
dtype: object
  code_magasin                 ville code_postal  surface_m2     ca_2024
0          M01       Bourg-en-Bresse       01000       420.5   812345.60
1          M02                  Nice       06000       615.0  1254300.25
2          M03  Charleville-Mézières       08000       380.0   598760.00
3          M04             Marseille       13001       890.5  2105400.90
4          M05              Toulouse       31000       720.0  1689005.10
5          M06                Rennes       35000       510.0  1102030.40
6          M07                 Lille       59000       655.5  1450120.75
7          M08                  Lyon       69001       940.0  2310880.30
code_magasin        str
ville               str
code_postal         str
surface_m2      float64
ca_2024         float64
dtype: object
```

In [64]:
brut_magasins = pd.read_csv("ressources/export_magasins.csv")
print(brut_magasins.shape)
print(list(brut_magasins.columns))

print(pd.read_csv("ressources/export_magasins.csv", sep=";").dtypes)

magasins = pd.read_csv(
    "ressources/export_magasins.csv",
    sep=";",
    decimal=",",
    dtype={"code_postal": str},
)
print(magasins)
print(magasins.dtypes)


(8, 1)
['code_magasin;ville;code_postal;surface_m2;ca_2024']
code_magasin      str
ville             str
code_postal     int64
surface_m2        str
ca_2024           str
dtype: object
  code_magasin                 ville code_postal  surface_m2     ca_2024
0          M01       Bourg-en-Bresse       01000       420.5   812345.60
1          M02                  Nice       06000       615.0  1254300.25
2          M03  Charleville-Mézières       08000       380.0   598760.00
3          M04             Marseille       13001       890.5  2105400.90
4          M05              Toulouse       31000       720.0  1689005.10
5          M06                Rennes       35000       510.0  1102030.40
6          M07                 Lille       59000       655.5  1450120.75
7          M08                  Lyon       69001       940.0  2310880.30
code_magasin        str
ville               str
code_postal         str
surface_m2      float64
ca_2024         float64
dtype: object


In [65]:
# Vérification
assert magasins.shape == (8, 5), "le fichier contient 8 magasins et 5 colonnes"
assert magasins.loc[1, "code_postal"] == "06000", "le code postal de Nice doit garder son zéro initial"
assert magasins["surface_m2"].dtype == "float64", "la surface doit être un nombre décimal : vérifie l'option decimal"
print("Question 4.4 : OK")


Question 4.4 : OK


**Réponse** : Sans option, pandas sépare les colonnes sur les virgules. L'en-tête n'en contient aucune : il est lu comme une seule colonne, dont le nom contient tous les en-têtes collés. Les lignes de données, elles, sont découpées sur leurs virgules décimales, et pandas range les morceaux en trop dans l'index : le résultat est inutilisable. Avec le seul séparateur `;`, les colonnes sont bonnes, mais `surface_m2` et `ca_2024` restent du texte, faute d'option `decimal`, et `code_postal` devient un entier qui perd son zéro initial (`1000` pour Bourg-en-Bresse).

### Question 4.5

Sur `magasins`, calcule le chiffre d'affaires total arrondi à l'euro et le chiffre d'affaires moyen par mètre carré, arrondi à 2 décimales (total des chiffres d'affaires divisé par total des surfaces).

**Résultat attendu** :

```text
11322843
2206.54
```

In [66]:
print(round(magasins["ca_2024"].sum()))
print(round(magasins["ca_2024"].sum() / magasins["surface_m2"].sum(), 2))


11322843
2206.54


### Question 4.6

Exporte `df` dans le dossier du TP en CSV (`ventes_export.csv`), en Excel (`ventes.xlsx`) et en Parquet (`ventes.parquet`), sans écrire l'index.

In [67]:
df.to_csv("ventes_export.csv", index=False)
df.to_excel("ventes.xlsx", index=False)
df.to_parquet("ventes.parquet", index=False)


In [68]:
# Vérification
from pathlib import Path

for nom in ["ventes_export.csv", "ventes.xlsx", "ventes.parquet"]:
    assert Path(nom).exists(), f"le fichier {nom} n'existe pas"
assert "Unnamed: 0" not in pd.read_csv("ventes_export.csv").columns, "le CSV ne doit pas contenir l'index"
print("Question 4.6 : OK")


Question 4.6 : OK


### Question 4.7

Relis les trois fichiers, sans aucune option, et affiche le type de la colonne `date` dans chacun. Quels formats ont conservé le type date ?

**Résultat attendu** :

```text
csv     : str
excel   : datetime64[us]
parquet : datetime64[us]
```

In [69]:
print("csv     :", pd.read_csv("ventes_export.csv")["date"].dtype)
print("excel   :", pd.read_excel("ventes.xlsx")["date"].dtype)
print("parquet :", pd.read_parquet("ventes.parquet")["date"].dtype)


csv     : str
excel   : datetime64[us]
parquet : datetime64[us]


**Réponse** : Le CSV est du texte : à la relecture, la date redevient une chaîne de caractères. Excel et Parquet enregistrent le type de chaque colonne et le restituent tel quel.

### Pour aller plus loin 4.8

Avec `pd.ExcelWriter`, écris dans un même fichier `ventes_par_feuille.xlsx` deux feuilles : `apercu`, avec les 10 premières lignes de `df`, et `complet`, avec toutes les lignes. Relis ensuite toutes les feuilles d'un coup avec `sheet_name=None`, et affiche le nom et la forme de chacune.

**Résultat attendu** :

```text
{'apercu': (10, 8), 'complet': (62, 8)}
```

In [70]:
with pd.ExcelWriter("ventes_par_feuille.xlsx") as writer:
    df.head(10).to_excel(writer, sheet_name="apercu", index=False)
    df.to_excel(writer, sheet_name="complet", index=False)

feuilles = pd.read_excel("ventes_par_feuille.xlsx", sheet_name=None)
{nom: feuille.shape for nom, feuille in feuilles.items()}


{'apercu': (10, 8), 'complet': (62, 8)}

## Partie 5 : exploration du jeu de données

*À faire après la section 7 du cours : première exploration d'un jeu de données.*

On travaille sur `df`, les ventes chargées avec la colonne `date` convertie.

### Question 5.1

Combien le jeu de données compte-t-il de lignes et de colonnes ? Affiche les premières et les dernières lignes, puis 5 lignes au hasard avec `random_state=0`.

In [71]:
print(df.shape)
display(df.head())
display(df.tail())
df.sample(5, random_state=0)


(62, 8)


,id_commande,date,region,categorie,produit,Prix unitaire,quantite,Client fidèle
0,1021,2025-01-01,Nouvelle-Aquitaine,Bureau,Lampe LED,268.78,9,False
1,1035,2025-01-02,Bretagne,Bureau,Rangement papier,316.64,3,False
2,1044,2025-01-05,Hauts-de-France,Électroménager,Grille-pain,248.61,10,False
3,1026,2025-01-11,Occitanie,Électroménager,Cafetière,92.78,2,False
4,1053,2025-01-13,Bretagne,Bureau,Bureau assis-debout,341.86,3,True


,id_commande,date,region,categorie,produit,Prix unitaire,quantite,Client fidèle
57,1006,2025-08-24,NaN,Papeterie,Classeur,214.47,1,False
58,1028,2025-09-06,Occitanie,Informatique,Écran 24 pouces,32.23,4,True
59,1036,2025-09-07,Hauts-de-France,Papeterie,Agrafeuse,105.56,12,True
60,1015,2025-09-10,Nouvelle-Aquitaine,Électroménager,Bouilloire,83.01,3,True
61,1020,2025-09-17,Hauts-de-France,Bureau,Chaise ergonomique,305.72,5,False


,id_commande,date,region,categorie,produit,Prix unitaire,quantite,Client fidèle
30,1010,2025-05-06,Nouvelle-Aquitaine,Papeterie,Bloc-notes,216.32,3,True
32,1032,2025-05-08,Bretagne,Électroménager,Cafetière,0.00,5,False
33,1017,2025-05-09,Nouvelle-Aquitaine,Électroménager,Bouilloire,211.79,8,False
48,1060,2025-07-17,Île-de-France,Papeterie,Classeur,157.25,9,False
51,1004,2025-08-05,Hauts-de-France,Papeterie,Bloc-notes,267.51,3,True


### Question 5.2

Utilise `info()`. Quelles colonnes contiennent des valeurs manquantes ? Confirme avec `isna().sum()`.

In [72]:
df.info()
df.isna().sum()


<class 'pandas.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id_commande    62 non-null     int64         
 1   date           62 non-null     datetime64[us]
 2   region         59 non-null     str           
 3   categorie      62 non-null     str           
 4   produit        62 non-null     str           
 5   Prix unitaire  62 non-null     float64       
 6   quantite       62 non-null     int64         
 7   Client fidèle  62 non-null     bool          
dtypes: bool(1), datetime64[us](1), float64(1), int64(2), str(3)
memory usage: 5.9 KB


id_commande      0
date             0
region           3
categorie        0
produit          0
Prix unitaire    0
quantite         0
Client fidèle    0
dtype: int64

**Réponse** : Seule la colonne `region` a moins de valeurs non nulles que de lignes : 59 sur 62. `isna().sum()` confirme 3 régions manquantes.

### Question 5.3

Utilise `describe()`, arrondi à 2 décimales. Regarde les minimums, les maximums, et compare la moyenne à la médiane (ligne `50%`). Quelles valeurs te semblent impossibles ou suspectes ?

In [73]:
df.describe().round(2)


,id_commande,date,Prix unitaire,quantite
count,62.00,62,62.00,62.00
mean,1030.89,2025-05-05 13:32:54.193548,406.80,13.85
min,1001.00,2025-01-01 00:00:00,0.00,-3.00
25%,1016.25,2025-03-01 00:00:00,114.80,3.00
50%,1031.50,2025-05-07 00:00:00,215.40,6.00
75%,1045.00,2025-07-09 00:00:00,283.84,9.00
max,1060.00,2025-09-17 00:00:00,12999.00,480.00
std,17.31,NaN,1628.35,60.27


**Réponse** : La quantité a un minimum à -3, impossible, et un maximum à 480 pour une médiane à 6, très suspect. Le prix unitaire a un minimum à 0 et un maximum à 12 999 : la moyenne (406,80) est presque le double de la médiane (215,40), tirée vers le haut par cette valeur extrême.

### Question 5.4

Affiche la répartition de la colonne `categorie` avec `value_counts`, puis celle de la colonne `region`, en comptant aussi les valeurs manquantes. Que remarques-tu sur la façon dont certaines valeurs sont écrites ?

**Résultat attendu** :

```text
categorie
Informatique      18
Électroménager    16
Bureau            14
Papeterie         13
INFORMATIQUE       1
Name: count, dtype: int64
region
Hauts-de-France         18
Nouvelle-Aquitaine      10
Île-de-France            9
Occitanie                8
Auvergne-Rhône-Alpes     7
Bretagne                 4
NaN                      3
Bretagne                 1
occitanie                1
Ile-de-France            1
Name: count, dtype: int64
```

In [74]:
print(df["categorie"].value_counts())
df["region"].value_counts(dropna=False)


categorie
Informatique      18
Électroménager    16
Bureau            14
Papeterie         13
INFORMATIQUE       1
Name: count, dtype: int64


region
Hauts-de-France         18
Nouvelle-Aquitaine      10
Île-de-France            9
Occitanie                8
Auvergne-Rhône-Alpes     7
Bretagne                 4
NaN                      3
Bretagne                 1
occitanie                1
Ile-de-France            1
Name: count, dtype: int64

**Réponse** : `INFORMATIQUE` est écrite en majuscules une fois, à côté de `Informatique`. Pour les régions, on trouve `occitanie` en minuscules, `Ile-de-France` sans accent et `Bretagne ` avec une espace finale, qui forment autant de valeurs distinctes, et 3 valeurs manquantes.

### Question 5.5

Affiche le nombre de valeurs distinctes de chaque colonne avec `nunique()`. Pour quelle colonne ce nombre devrait-il être égal au nombre de lignes ?

**Résultat attendu** :

```text
id_commande      60
date             53
region            9
categorie         5
produit          16
Prix unitaire    60
quantite         14
Client fidèle     2
dtype: int64
```

In [75]:
df.nunique()


id_commande      60
date             53
region            9
categorie         5
produit          16
Prix unitaire    60
quantite         14
Client fidèle     2
dtype: int64

**Réponse** : `id_commande` identifie une commande : elle devrait avoir 62 valeurs distinctes, une par ligne. Elle n'en a que 60.

### Question 5.6

Trouve les commandes qui apparaissent plusieurs fois : affiche les 3 premières valeurs de `df["id_commande"].value_counts()`, puis toutes les lignes de ces commandes avec `isin`.

In [76]:
repetitions = df["id_commande"].value_counts()
print(repetitions.head(3))
df[df["id_commande"].isin([1045, 1040])]


id_commande
1045    2
1040    2
1021    1
Name: count, dtype: int64


,id_commande,date,region,categorie,produit,Prix unitaire,quantite,Client fidèle
15,1045,2025-03-01,Occitanie,Papeterie,Bloc-notes,329.11,3,False
16,1045,2025-03-01,Occitanie,Papeterie,Bloc-notes,329.11,3,False
39,1040,2025-06-11,Hauts-de-France,Bureau,Chaise ergonomique,219.83,7,False
40,1040,2025-06-11,Hauts-de-France,Bureau,Chaise ergonomique,219.83,7,False


**Réponse** : Les commandes 1045 et 1040 apparaissent deux fois, avec des lignes strictement identiques : ce sont des doublons à supprimer au nettoyage.

### Question 5.7

Réponds en une ligne de code chacune :

1. Quelle proportion des commandes vient de clients fidèles ? (`value_counts(normalize=True)`)
2. Quel est le produit le plus souvent commandé, et combien de fois ?
3. Quelle période couvrent les données ? (`min` et `max` de la colonne `date`)

**Résultat attendu** :

```text
Client fidèle
False    0.629
True     0.371
Name: proportion, dtype: float64
produit
Souris sans fil    10
Name: count, dtype: int64
2025-01-01 00:00:00 2025-09-17 00:00:00
```

In [77]:
print(df["Client fidèle"].value_counts(normalize=True).round(3))
print(df["produit"].value_counts().head(1))
print(df["date"].min(), df["date"].max())


Client fidèle
False    0.629
True     0.371
Name: proportion, dtype: float64
produit
Souris sans fil    10
Name: count, dtype: int64
2025-01-01 00:00:00 2025-09-17 00:00:00


### Question 5.8

Utilise les commandes de ton choix pour chercher d'autres anomalies, par exemple dans les noms de colonnes ou dans les associations entre catégorie et produit : essaie `df[["categorie", "produit"]].value_counts()`.

In [78]:
print(list(df.columns))
df[["categorie", "produit"]].value_counts().sort_index()


['id_commande', 'date', 'region', 'categorie', 'produit', 'Prix unitaire', 'quantite', 'Client fidèle']


categorie       produit            
Bureau          Bureau assis-debout    4
                Chaise ergonomique     4
                Lampe LED              4
                Rangement papier       2
INFORMATIQUE    Souris sans fil        1
Informatique    Casque audio           2
                Clavier mécanique      2
                Souris sans fil        9
                Webcam HD              2
                Écran 24 pouces        3
Papeterie       Agrafeuse              3
                Bloc-notes             4
                Classeur               2
                Stylo plume            4
Électroménager  Bouilloire             7
                Cafetière              5
                Grille-pain            4
Name: count, dtype: int64

**Réponse** : Les noms de colonnes `Prix unitaire` et `Client fidèle` contiennent des majuscules, une espace et un accent, contrairement aux autres. Chaque produit est rattaché à une seule catégorie, à l'exception de la `Souris sans fil` classée une fois en `INFORMATIQUE`.

## Synthèse de l'exploration

**Dimensions** : 62 lignes et 8 colonnes, des ventes de janvier à septembre 2025.

**Types à corriger** : la colonne `date` est lue comme du texte sans l'option `parse_dates` ; `categorie` et `region`, qui prennent peu de valeurs, pourront passer en type `category` une fois nettoyées.

**Valeurs manquantes** : 3 régions manquantes (commandes 1018, 1043 et 1006).

**Valeurs aberrantes repérées** : une quantité négative (-3, commande 1030), une quantité de 480 (commande 1051) pour une médiane à 6, un prix nul (commande 1032) et un prix de 12 999 euros pour une souris sans fil (commande 1016).

**Incohérences d'écriture** : `INFORMATIQUE` en majuscules ; `occitanie` en minuscules, `Ile-de-France` sans accent, `Bretagne ` avec une espace finale.

**Autres anomalies** : 2 lignes en double (commandes 1045 et 1040) ; noms de colonnes irréguliers (`Prix unitaire`, `Client fidèle`).

**Questions à creuser** : la quantité de 480 est-elle une vraie commande d'entreprise ou une erreur de saisie ? Que signifie un prix nul : un geste commercial, une erreur ? Les régions manquantes peuvent-elles être retrouvées à partir d'une autre source ?

## Avant de rendre

- Le fichier `requirements.txt` est présent dans le dossier du TP.
- Toutes les cellules de vérification affichent `OK`.
- Le notebook s'exécute sans erreur de haut en bas : menu *Kernel*, puis *Restart Kernel and Run All Cells*.